# Classificação Fine-Grained de Raças de Cães

**Trabalho final — Visão Computacional**
Pós-graduação em LLM e IA Generativa

Autores: Victor Macaubas e Mari
Data: agosto/2026

Repositório: https://github.com/victormacaubas/project-image-processing

---

> **Este notebook é autossuficiente e roda em ~2 minutos.**
>
> Basta `Ambiente de execução → Executar tudo`. A primeira célula clona o
> repositório e instala o que falta; nenhum arquivo adicional é necessário e
> nada precisa ser baixado.
>
> Os resultados, gráficos e a análise de erros são reconstruídos a partir das
> predições (logits) versionadas no repositório — por isso é rápido.
>
> Para reexecutar os treinos do zero, troque `RETRAIN` para `True` na célula de
> setup. Aí sim leva ~2 horas com GPU.

In [ ]:
# ─── Bootstrap ──────────────────────────────────────────────────────────────
# Deixa o notebook autossuficiente: clona o repositório e instala o que falta.
# Idempotente — rodar duas vezes não causa dano.
#
# NÃO reinstala torch/torchvision: o Colab já os traz, e reinstalar dispara
# "restart runtime", que aborta a execução de ponta a ponta.

REPO_URL = "https://github.com/victormacaubas/project-image-processing.git"
REPO_NAME = "project-image-processing"

import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def _run(cmd: list[str]) -> None:
    """Roda o comando e, se falhar, mostra a saída real.

    Sem isso, um pip que falha com -q some silenciosamente e o erro só
    aparece 20 células depois, como ImportError sem contexto.
    """
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout)
        print(result.stderr, file=sys.stderr)
        raise RuntimeError(f"Falhou: {' '.join(cmd)}")


def _find_repo_root() -> Path:
    """Localiza a raiz do repo, clonando se necessário. Idempotente."""
    here = Path.cwd()

    # Já estamos dentro do repositório?
    for candidate in (here, *here.parents):
        if (candidate / "src" / "dogs" / "config.py").exists():
            return candidate

    # Já clonado num subdiretório?
    if (here / REPO_NAME / "src" / "dogs" / "config.py").exists():
        return here / REPO_NAME

    # Clonar.
    print(f"Clonando {REPO_URL} ...")
    _run(["git", "clone", "--depth", "1", REPO_URL, REPO_NAME])
    return here / REPO_NAME


REPO_ROOT = _find_repo_root()

if IN_COLAB:
    reqs = REPO_ROOT / "requirements-colab.txt"
    if reqs.exists():
        print("Instalando dependências ...")
        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(reqs)])

# Torna o pacote `dogs` importável, sem duplicar entradas em sys.path.
src = str(REPO_ROOT / "src")
if src not in sys.path:
    sys.path.insert(0, src)

print(f"Repositório: {REPO_ROOT}")

In [ ]:
# ─── Verificação ────────────────────────────────────────────────────────────
# Falha aqui, alto e claro, em vez de estourar um erro críptico 20 células
# adiante. Se esta célula passar, o notebook roda até o fim.

import importlib

problemas = []

for pacote in ["torch", "torchvision", "numpy", "pandas", "sklearn",
               "matplotlib", "seaborn", "datasets"]:
    try:
        importlib.import_module(pacote)
    except ImportError as erro:
        problemas.append(f"pacote ausente: {pacote} ({erro})")

try:
    from dogs.config import describe_environment, ensure_dirs
    ensure_dirs()
    print(describe_environment())
except Exception as erro:
    problemas.append(f"pacote `dogs` não importável: {erro}")

if problemas:
    raise RuntimeError(
        "Ambiente incompleto:\n  - " + "\n  - ".join(problemas)
        + "\n\nRode a célula de bootstrap acima antes desta."
    )

print("\nAmbiente OK.")

In [ ]:
# ─── Setup ──────────────────────────────────────────────────────────────────
# False -> reconstrói tudo das predições versionadas (~2 min). É o padrão.
# True  -> gera EDA, E1 e E2 a partir do dataset (GPU para E1).
RETRAIN = False

# E3 já foi avaliado no teste. Só ative para uma reexecução deliberada do E3.
RETRAIN_E3 = False

import logging
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns

from dogs.config import (TrainConfig, FEATURES_DIR, CHECKPOINT_DIR,
                         PREDICTIONS_DIR, RESULTS_CSV)

# force=True é obrigatório: o Colab instala um handler no logger raiz antes desta
# célula, e sem isso o basicConfig sai calado, o nível fica em WARNING e nenhum log
# de progresso do treino aparece.
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", force=True)
torch.manual_seed(42)
np.random.seed(42)

### Artefatos

Este notebook é **o documento de trabalho e a entrega ao mesmo tempo**. Durante a
semana ele fica incompleto; a célula abaixo mostra o que já existe e o que falta.

| Nível | Precisa de | Vem de | Tempo |
|---|---|---|---|
| **Padrão** (`RETRAIN = False`) | predições (~3 MB cada) | repositório | ~2 min |
| Reproduzir os treinos | embeddings + checkpoints | Drive | ~2 h |

No nível padrão nada é baixado: os logits de cada experimento estão versionados
no Git, e deles saem a tabela, os gráficos e a análise de erros. Um checkpoint de
ResNet50 tem ~100 MB; os logits que ele produz, ~3 MB — e para refazer a
*análise*, os logits bastam.

In [ ]:
# ─── Estado dos experimentos ────────────────────────────────────────────────
# Durante a semana isto é o painel de progresso. Na sexta, precisa estar todo OK.
DRIVE_FOLDER_ID = "12XC2LBi7NK02kTdIMN6FAU_Wf7sbKYZ8"

from pathlib import Path

# Cada entrada: (nome do experimento, split, quem faz)
ESPERADOS = [
    ("E1_scratch",        "val",  "Mari"),
    ("E2_linear_probe",   "val",  "Mari"),
    ("E2_linear_probe_sem_bn", "val", "Mari"),
    ("E3_finetune_last2", "val",  "Victor"),
    ("E3_finetune_last2", "test", "Victor"),   # só na sexta
]

ASSINATURAS = {".npy": b"\x93NUMPY", ".npz": b"PK", ".pt": b"PK"}


def arquivo_valido(caminho: Path) -> bool:
    """Detecta HTML gravado com extensão de dados.

    O Google intercepta downloads grandes com uma página de aviso de vírus; sem
    esta checagem o gdown grava esse HTML como .npy, e o erro só aparece muito
    depois como um "corrupt file" incompreensível.
    """
    esperado = ASSINATURAS.get(caminho.suffix)
    if esperado is None:
        return True
    with caminho.open("rb") as f:
        return f.read(len(esperado)) == esperado


def baixar_artefatos() -> None:
    import gdown
    destino = FEATURES_DIR.parent
    gdown.download_folder(id=DRIVE_FOLDER_ID, output=str(destino),
                          quiet=False, use_cookies=False)

    arquivos_baixados = [p for p in destino.rglob("*") if p.is_file()]
    if not arquivos_baixados:
        raise RuntimeError(
            "O download do Drive não trouxe nenhum arquivo de dados.\n"
            "As subpastas são recriadas mesmo vazias, então isto passa silenciosamente "
            "e só estoura células adiante como FileNotFoundError.\n"
            "Confira: (1) os .npy estão mesmo na pasta do Drive; (2) o "
            "compartilhamento 'Qualquer pessoa com o link' vale para os arquivos, "
            "não apenas para a pasta."
        )

    corrompidos = [
        p for p in arquivos_baixados if p.suffix in ASSINATURAS and not arquivo_valido(p)
    ]
    if corrompidos:
        for p in corrompidos:
            p.unlink()
        raise RuntimeError(
            f"{len(corrompidos)} arquivo(s) baixados como HTML, não como dados.\n"
            "Causa provável: a pasta do Drive não está pública, ou o Google "
            "interceptou o download (comum acima de 100 MB).\n"
            "Saídas: (1) conferir acesso 'Qualquer pessoa com o link'; "
            "(2) publicar numa GitHub Release; (3) rodar com RETRAIN = False."
        )


EMBEDDINGS_E2 = [
    FEATURES_DIR / f"resnet50_{split}_{sufixo}.npy"
    for split in ("train", "val")
    for sufixo in ("X", "y")
]

if RETRAIN:
    faltando_embeddings = [p.name for p in EMBEDDINGS_E2 if not p.exists()]
    if faltando_embeddings:
        baixar_artefatos()
        faltando_embeddings = [p.name for p in EMBEDDINGS_E2 if not p.exists()]
    if faltando_embeddings:
        print("Embeddings ausentes no Drive; gerando treino e validação com ResNet50.")
        from dogs.features import ensure_feature_splits

        ensure_feature_splits("resnet50", splits=("train", "val"))
        faltando_embeddings = [p.name for p in EMBEDDINGS_E2 if not p.exists()]
    if faltando_embeddings:
        raise RuntimeError(f"Falha ao gerar embeddings: {faltando_embeddings}")

# ─── Painel ─────────────────────────────────────────────────────────────────
print(f"{'experimento':22s} {'split':6s} {'dono':7s} estado")
print("─" * 58)

faltando = []
for nome, split, dono in ESPERADOS:
    caminho = PREDICTIONS_DIR / f"{nome}_{split}.npz"
    if caminho.exists():
        estado = f"ok  ({caminho.stat().st_size / 1e6:.1f} MB)"
    else:
        estado = "FALTA"
        faltando.append(f"{nome}/{split}")
    print(f"{nome:22s} {split:6s} {dono:7s} {estado}")

print()
if faltando:
    print(f"Faltam {len(faltando)}: {', '.join(faltando)}")
    print("Normal durante a semana. Na sexta, tem que estar tudo ok.")
else:
    print("Todos os experimentos presentes — pronto para o teste de entrega.")

### Bastidor

Operações que usamos durante o desenvolvimento e que **não fazem parte da
entrega**: montar o Drive, gerar embeddings, subir artefatos.

Ficam atrás de `MODO_TRABALHO` em vez de serem apagadas — assim ninguém esquece
de removê-las na sexta, e quem for reproduzir o trabalho vê o que foi feito.

Com `MODO_TRABALHO = False` (padrão) esta célula não faz nada: quem estiver
corrigindo não recebe pedido de autorização do Google Drive.

In [ ]:
MODO_TRABALHO = False   # True só enquanto estamos desenvolvendo

if MODO_TRABALHO:
    from pathlib import Path

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE = Path('/content/drive/MyDrive/project-image-processing')
    print("features/   ", (DRIVE / 'features').exists())
    print("checkpoints/", (DRIVE / 'checkpoints').exists())

    # ── Descomente conforme a tarefa do dia ──────────────────────────────

    # Teste de fumaça do data.py. Baixa 776 MB na primeira vez (5-10 min).
    # Esperado: [64, 3, 224, 224] · [64] · 120 · n02085620-Chihuahua
    #
    # from dogs.data import load_data
    # dados = load_data(TrainConfig(experiment_name="smoke"))
    # imagens, rotulos = next(iter(dados.train_loader))
    # print(imagens.shape, rotulos.shape, dados.num_classes)
    # print(dados.class_names[:3])

    # Gerar os embeddings (~8 min na T4, roda uma vez):
    # !python -m dogs.features --backbone resnet50

    # Subir para o Drive, e avisar a Mari:
    # !cp data/processed/features/*.npy "{DRIVE}/features/"
    # !ls -lh "{DRIVE}/features/"

---
# 1. Descrição do problema

O objetivo deste trabalho é classificar, a partir de uma fotografia, a raça de um cão entre 120 possibilidades. Trata-se de um problema de *classificação fine-grained*: ao contrário de uma classificação genérica, em que as classes podem ser visualmente distantes (por exemplo, cão, carro e avião), todas as categorias aqui pertencem ao mesmo grupo semântico. A decisão depende, portanto, de evidências sutis como formato do focinho, textura e cor da pelagem, proporção das orelhas e estrutura corporal.

A tarefa é difícil por combinar baixa variância entre classes com alta variância dentro da própria classe. Pose, iluminação, idade do animal, fundo, oclusões e enquadramento podem tornar duas imagens da mesma raça visualmente mais diferentes entre si do que imagens de duas raças próximas. Essa situação aparece em aplicações reais de catalogação de animais, apoio a abrigos e clínicas veterinárias, organização de acervos fotográficos e interfaces de busca por imagem.

> **Pergunta que guia o trabalho: quanto de representação visual é preciso aprender, em vez de transferir, para resolver classificação fine-grained com dados limitados?**

Para respondê-la, comparamos uma CNN pequena treinada do zero, um classificador linear sobre embeddings de uma ResNet50 congelada e o fine-tuning parcial da mesma arquitetura. Consideramos sucesso não apenas obter alta acurácia top-1, mas também mostrar uma progressão coerente entre os experimentos, boa acurácia top-5 e F1 macro compatível com o desempenho global, além de analisar os erros mais informativos.

---
# 2. Descrição da base de dados

Usamos o Stanford Dogs, proposto por Khosla et al. para classificação subordinada de raças. A base reúne 20.580 imagens de 120 raças, com split oficial de 12.000 imagens de treino e 8.580 de teste. A partir do treino oficial, reservamos 15% para validação com semente fixa `SEED = 42`, obtendo 10.200 imagens de treino e 1.800 de validação. O teste permanece separado e é usado somente na avaliação final do melhor modelo. Por derivar de imagens do ImageNet, a utilização segue os termos de pesquisa e educação não comercial do ImageNet; ele não detém os direitos autorais individuais das imagens.

O enunciado indicava `Voxel51/StanfordDogs`, mas esse repositório está no formato FiftyOne: `load_dataset()` não falha, porém retorna as 20.580 imagens em um único split e sem a coluna de rótulos, uma falha silenciosa que só apareceria no treinamento. Por isso adotamos `maurice-fp/stanford-dogs`, versão em parquet que preserva o split oficial, a coluna `label` como `ClassLabel` e os nomes WordNet das raças, como `n02085620-Chihuahua`. Essa escolha foi uma decisão de engenharia de dados para garantir rótulos, reprodutibilidade e validação correta antes de qualquer treinamento.

A Figura 1 apresenta a distribuição de imagens por classe, incluindo mínimo, máximo e mediana impressos durante a geração; ela orienta a leitura conjunta de top-1 e F1 macro. A Figura 2 mostra amostras cruas de dez raças, a Figura 3 evidencia a variedade de proporções e resoluções originais que motiva o redimensionamento para 224×224, e a Figura 4 registra pares visualmente próximos escolhidos antes do treino. Como o Stanford Dogs foi construído a partir do ImageNet, essa origem também é uma ameaça importante à validade externa e é discutida na Seção 6.

In [ ]:
# ─── Análise exploratória ──────────────────────────────────────────────────
# Com RETRAIN=True, gera e versiona os artefatos. Na entrega, só os exibe.
import json

from IPython.display import Image, display

from dogs.config import FIGURES_DIR, REPORTS_DIR

NOMES_FIGURAS_EDA = [
    "distribuicao_classes",
    "grid_amostras",
    "resolucoes",
    "pares_parecidos",
]

if RETRAIN:
    from dogs.data import _load_raw
    from dogs.eda import (
        dispersao_resolucoes,
        distribuicao_classes,
        grid_pares_parecidos,
        indices_por_classe,
    )
    from dogs.viz import grid_de_amostras

    dataset_raw = _load_raw()
    class_names = list(dataset_raw["train"].features["label"].names)
    assert len(class_names) == 120
    (REPORTS_DIR / "class_names.json").write_text(
        json.dumps(class_names, ensure_ascii=False, indent=0), encoding="utf-8"
    )

    _, estatisticas_classes = distribuicao_classes(
        dataset_raw["train"]["label"],
        class_names,
        salvar_em=FIGURES_DIR / "distribuicao_classes.png",
    )
    print("Imagens por classe — " + ", ".join(
        f"{nome}: {valor:.0f}" for nome, valor in estatisticas_classes.items()
    ))

    classes_amostra = np.linspace(0, len(class_names) - 1, num=10, dtype=int)
    grid_de_amostras(
        dataset_raw["train"],
        indices_por_classe(dataset_raw["train"], classes_amostra),
        class_names,
        n_colunas=4,
        titulo="Amostras de raças do Stanford Dogs",
        salvar_em=FIGURES_DIR / "grid_amostras.png",
    )
    dispersao_resolucoes(
        dataset_raw["train"], salvar_em=FIGURES_DIR / "resolucoes.png"
    )
    pares_apostados = [
        ("n02093754-Border_terrier", "n02095889-Sealyham_terrier"),
        ("n02093991-Irish_terrier", "n02094114-Norfolk_terrier"),
        ("n02110063-malamute", "n02110185-Siberian_husky"),
        ("n02113023-Pembroke", "n02113186-Cardigan"),
    ]
    grid_pares_parecidos(
        dataset_raw["train"],
        class_names,
        pares_apostados,
        salvar_em=FIGURES_DIR / "pares_parecidos.png",
    )
else:
    for nome in NOMES_FIGURAS_EDA:
        caminho = FIGURES_DIR / f"{nome}.png"
        if not caminho.exists():
            raise FileNotFoundError(
                f"Figura da EDA ausente: {caminho.name}. Rode uma vez com RETRAIN=True."
            )
        display(Image(filename=str(caminho)))

---
# 3. Metodologia

A metodologia progride de uma representação aprendida inteiramente com a base para uma representação transferida e, por fim, adaptada. O E1 usa uma CNN pequena treinada do zero, formada por quatro blocos `Conv2d(3×3) → BatchNorm2d → ReLU → MaxPool2d(2)`, com 32, 64, 128 e 256 canais. Um `AdaptiveAvgPool2d(1)` reduz a saída a 256 atributos, seguidos de dropout e uma camada linear para as 120 classes. A arquitetura possui 420.216 parâmetros — valor calculado pela própria implementação, em vez da estimativa de aproximadamente um milhão do roteiro.

No E2, congelamos uma ResNet50 pré-treinada e treinamos apenas `BatchNorm1d → Linear` sobre seus embeddings de 2.048 dimensões. A normalização é habilitada e desabilitada em uma ablação: como as features vêm após uma ReLU, são não negativas e têm escalas distintas entre dimensões; normalizá-las melhora o condicionamento do problema para o classificador linear. No E3, substituímos a cabeça da ResNet50 e descongelamos somente os dois últimos blocos, permitindo adaptar features de alto nível sem atualizar todo o backbone.

As imagens são convertidas para RGB, redimensionadas e normalizadas com média e desvio-padrão do ImageNet. Apenas o conjunto de treino recebe `RandomResizedCrop`, espelhamento horizontal e pequenas variações de cor; validação e teste usam `Resize` seguido de `CenterCrop`, para que a avaliação seja determinística e não introduza transformações aleatórias. Todos os experimentos usam AdamW, scheduler cosseno, label smoothing de 0,1, precisão mista quando há GPU e early stopping com paciência quatro, preservando o checkpoint de maior top-1 na validação.

A métrica principal é a acurácia top-1. A top-5 é relevante em 120 classes fine-grained porque informa se a raça correta ficou entre hipóteses visualmente plausíveis; o F1 macro impede que um bom desempenho em classes mais frequentes esconda falhas em classes menores. Para reduzir custo computacional, os embeddings da ResNet50 foram pré-computados uma única vez: a extração sobre as imagens custa minutos em GPU, mas os classificadores lineares posteriores treinam em segundos na CPU. E1 e E2 são comparados exclusivamente na validação; o split de teste é reservado para uma única avaliação do E3.

## 3.1 Extração de embeddings

Passo executado uma vez, fora do notebook:

```bash
python -m dogs.features --backbone resnet50
```

Gera `{backbone}_{split}_{X,y}.npy` em `data/processed/features/`.

In [ ]:
# Os embeddings (~170 MB) não são versionados, e só o treino do linear probe precisa
# deles. Com RETRAIN = False o E2 sai dos logits salvos, como os outros experimentos.
if RETRAIN:
    from dogs.features import load_features

    X_train, y_train = load_features(split="train")
    X_val, y_val = load_features(split="val")
    print(X_train.shape, X_val.shape)
else:
    print("Embeddings não carregados (RETRAIN = False); o E2 vem dos logits salvos.")


---
# 4. Experimentos

<!-- OBRIGATÓRIO NA ENTREGA -->

Cada experimento: o que testa, como foi configurado, o que aconteceu.

## E1 — CNN treinada do zero

**Hipótese:** sem transferência, 120 classes fine-grained com ~100 imagens de treino
por classe não dão sinal suficiente. Esperamos acurácia baixa.

*Dona: Mari*

In [ ]:
from dogs.evaluate import load_predictions, metrics_from_logits
from dogs.models import SmallCNN

NOME = "E1_scratch"

if RETRAIN:
    from dogs.data import load_data
    from dogs.evaluate import log_result, predict, save_predictions
    from dogs.train import get_device, load_checkpoint, train_model

    config_e1 = TrainConfig(
        experiment_name=NOME, num_epochs=15, learning_rate=3e-4
    )
    model_e1 = SmallCNN()
    saida_sanity = model_e1(torch.randn(2, 3, 224, 224))
    assert saida_sanity.shape == (2, 120)
    print(f"sanidade: {tuple(saida_sanity.shape)}, "
          f"{sum(p.numel() for p in model_e1.parameters()):,} parâmetros")

    dados_e1 = load_data(config_e1)
    print(train_model(model_e1, dados_e1.train_loader, dados_e1.val_loader, config_e1))
    model_e1 = load_checkpoint(model_e1, config_e1)
    logits, labels = predict(model_e1, dados_e1.val_loader, get_device())
    metricas_e1 = metrics_from_logits(logits, labels)
    log_result(NOME, "val", metricas_e1)
    save_predictions(NOME, "val", logits, labels)
    print(f"val:  {metricas_e1}")
else:
    logits, labels = load_predictions(NOME, "val")
    print(f"val:  {metrics_from_logits(logits, labels)}")


## E2 — Linear probe sobre backbone congelado

**Hipótese:** a representação do ImageNet já separa bem as raças, mesmo sem nenhuma
adaptação — um classificador linear deve superar E1 por larga margem.

*Dona: Mari*

In [ ]:
from dogs.evaluate import load_predictions, metrics_from_logits
from dogs.models import LinearProbe

NOME = "E2_linear_probe"

if RETRAIN:
    from dogs.evaluate import log_result, predict, save_predictions
    from dogs.features import load_features, make_feature_loader
    from dogs.train import get_device, load_checkpoint, train_model

    X_train, y_train = load_features(split="train")
    X_val, y_val = load_features(split="val")
    treino_e2 = make_feature_loader(X_train, y_train, shuffle=True)
    validacao_e2 = make_feature_loader(X_val, y_val, shuffle=False)

    for nome_experimento, usar_batchnorm in [
        ("E2_linear_probe", True),
        ("E2_linear_probe_sem_bn", False),
    ]:
        model_e2 = LinearProbe(X_train.shape[1], use_batchnorm=usar_batchnorm)
        saida_sanity = model_e2(torch.randn(4, X_train.shape[1]))
        assert saida_sanity.shape == (4, 120)

        config_e2 = TrainConfig(
            experiment_name=nome_experimento, num_epochs=30, learning_rate=1e-3
        )
        print(train_model(model_e2, treino_e2, validacao_e2, config_e2))
        model_e2 = load_checkpoint(model_e2, config_e2)
        logits, labels = predict(model_e2, validacao_e2, get_device())
        metricas_e2 = metrics_from_logits(logits, labels)
        log_result(nome_experimento, "val", metricas_e2)
        save_predictions(nome_experimento, "val", logits, labels)
        print(f"{nome_experimento}: {metricas_e2}")
else:
    for nome_experimento in (NOME, "E2_linear_probe_sem_bn"):
        logits, labels = load_predictions(nome_experimento, "val")
        print(f"{nome_experimento}: {metrics_from_logits(logits, labels)}")


## E3 — Fine-tuning parcial

**Hipótese:** descongelar os blocos finais permite adaptar as features de alto nível
ao domínio e deve superar E2.

*Dono: Victor*

In [ ]:
import json

from dogs.config import REPORTS_DIR, TrainConfig
from dogs.data import load_data
from dogs.evaluate import (
    load_predictions,
    log_result,
    metrics_from_logits,
    predict,
    save_predictions,
)
from dogs.models import build_finetune_model
from dogs.train import get_device, load_checkpoint, train_model

config = TrainConfig(
    experiment_name="E3_finetune_last2",
    unfreeze_last_n_blocks=2,
    learning_rate=1e-4,
    num_epochs=15,
)

if RETRAIN_E3:
    data = load_data(config)
    class_names = data.class_names

    model = build_finetune_model("resnet50", config.unfreeze_last_n_blocks)
    train_model(model, data.train_loader, data.val_loader, config)
    model = load_checkpoint(model, config)

    device = get_device()
    logits_val, labels_val = predict(model, data.val_loader, device)
    logits_test, labels_test = predict(model, data.test_loader, device)
    metrics_val = metrics_from_logits(logits_val, labels_val)
    metrics_test = metrics_from_logits(logits_test, labels_test)

    for split, logits, labels, metrics in [
        ("val", logits_val, labels_val, metrics_val),
        ("test", logits_test, labels_test, metrics_test),
    ]:
        log_result(config.experiment_name, split, metrics)
        save_predictions(config.experiment_name, split, logits, labels)
else:
    class_names = json.loads(
        (REPORTS_DIR / "class_names.json").read_text(encoding="utf-8")
    )

    logits_val, labels_val = load_predictions(config.experiment_name, "val")
    logits_test, labels_test = load_predictions(config.experiment_name, "test")
    metrics_val = metrics_from_logits(logits_val, labels_val)
    metrics_test = metrics_from_logits(logits_test, labels_test)

    # Checkpoint (~100 MB) nao e versionado no repo; carrega so se existir localmente.
    if config.checkpoint_path().exists():
        model = build_finetune_model("resnet50", config.unfreeze_last_n_blocks)
        model = load_checkpoint(model, config)

print(f"val:  {metrics_val}")
print(f"test: {metrics_test}")

---
# 5. Resultados

<!-- OBRIGATÓRIO NA ENTREGA -->

In [ ]:
results = pd.read_csv(RESULTS_CSV)
results.sort_values("top1", ascending=False)

In [ ]:
from dogs.config import FIGURES_DIR
from dogs.viz import comparar_experimentos

# Só o split de validação: é o único que todos os experimentos têm, e dois splits do
# mesmo experimento viram barras sobrepostas no eixo categórico.
comparar_experimentos(
    results[results["split"] == "val"], salvar_em=FIGURES_DIR / "comparacao_top1.png"
)


---
# 6. Análise

<!-- OBRIGATÓRIO NA ENTREGA — dono: Victor, prazo: quinta -->

## 6.1 O salto do transfer learning

Comparar E1 vs. E2 vs. E3 e interpretar a magnitude da diferença.

## 6.2 Quais raças o modelo confunde

Pares mais confundidos. As confusões são visualmente plausíveis? Um humano erraria
os mesmos casos?

## 6.3 ⚠️ Contaminação entre Stanford Dogs e ImageNet

Stanford Dogs foi construído a partir do ImageNet. O backbone pré-treinado em
ImageNet-1k **já viu essas imagens**. Nossos números de transfer learning são,
portanto, otimistas e não estimam o desempenho em um domínio novo.

Discutir: o que isso invalida, o que continua válido, e como um experimento futuro
poderia medir o efeito (ex.: avaliar em fotos de cães fora do ImageNet).

## 6.4 Limitações

Orçamento computacional, ausência de busca de hiperparâmetros, execução única
por experimento (sem barras de erro), split de teste tocado uma só vez.

In [ ]:
from dogs.config import FIGURES_DIR
from dogs.evaluate import most_confused_pairs
from dogs.viz import (
    grid_pares_confundidos,
    grid_piores_erros,
    matriz_confusao_recorte,
    nome_legivel,
)

pares = most_confused_pairs(logits_test, labels_test, class_names, top_n=10)
for real, previsto, contagem in pares:
    print(f"{nome_legivel(real):25s} -> {nome_legivel(previsto):25s} {contagem}x")

# Recorte a partir dos 5 pares mais frequentes: 10 pares já renderiam uma matriz de
# ~20 classes, tão ilegível quanto a 120x120 que o recorte existe para evitar.
nomes_unicos = list(
    dict.fromkeys(nome for real, previsto, _ in pares[:5] for nome in (real, previsto))
)
indices_classes = [class_names.index(nome) for nome in nomes_unicos]

matriz_confusao_recorte(
    logits_test,
    labels_test,
    class_names,
    indices_classes,
    salvar_em=FIGURES_DIR / "matriz_confusao_recorte.png",
)

if RETRAIN:
    from dogs.data import _load_raw

    test_dataset = _load_raw()["test"]
    grid_pares_confundidos(
        test_dataset,
        logits_test,
        labels_test,
        class_names,
        top_n=5,
        salvar_em=FIGURES_DIR / "pares_confundidos.png",
    )
    grid_piores_erros(
        test_dataset,
        logits_test,
        labels_test,
        class_names,
        top_n=10,
        salvar_em=FIGURES_DIR / "piores_erros.png",
    )
else:
    from IPython.display import Image, display

    for nome in ["pares_confundidos", "piores_erros"]:
        caminho = FIGURES_DIR / f"{nome}.png"
        if not caminho.exists():
            raise FileNotFoundError(
                f"Figura da análise ausente: {caminho.name}. Rode uma vez com RETRAIN=True."
            )
        display(Image(filename=str(caminho)))

---
# 7. Conclusões

<!-- OBRIGATÓRIO NA ENTREGA — dona: Mari, prazo: sexta -->

**Escrever aqui:**

- Resposta direta à pergunta da seção 1
- O que os números mostraram, incluindo o que surpreendeu
- O que faríamos com mais tempo (E5, E6, métodos fine-grained com atenção por partes)

---
## Referências

- Khosla et al. (2011). *Novel Dataset for Fine-Grained Image Categorization: Stanford Dogs.*
- He et al. (2016). *Deep Residual Learning for Image Recognition.*
- Radford et al. (2021). *Learning Transferable Visual Models From Natural Language Supervision.*